In [ ]:
## Ejecutar para descargar los datos
!wget -P ./models https://cs.famaf.unc.edu.ar/~ccardellino/SBWCE/SBW-vectors-300-min5.bin.gz && gunzip ./models/SBW-vectors-300-min5.bin.gz
!pip install gensim

--2026-04-13 14:11:13--  https://cs.famaf.unc.edu.ar/~ccardellino/SBWCE/SBW-vectors-300-min5.bin.gz
Resolving cs.famaf.unc.edu.ar (cs.famaf.unc.edu.ar)... 200.16.17.55
Connecting to cs.famaf.unc.edu.ar (cs.famaf.unc.edu.ar)|200.16.17.55|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1123304474 (1.0G) [application/x-gzip]
Saving to: ‘./models/SBW-vectors-300-min5.bin.gz’

SBW-vectors-300-min 100%[===================>]   1.05G  17.8MB/s    in 60s     

2026-04-13 14:12:14 (17.9 MB/s) - ‘./models/SBW-vectors-300-min5.bin.gz’ saved [1123304474/1123304474]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 80.0 MB/s eta 0:00:00


![](https://www.unsam.edu.ar/img/logo_ecyt.png)


# **Licenciatura en Ciencia de Datos**
# _Materia: Introducción a NLP_

# Clase 5c. Clasificación de textos con diferentes features



# Introducción

El objetivo de la clase de hoy es avanzar en el entrenamiento de un modelo de clasificación de texto. Para ello, vamos a utilizar un dataset de reseñas que hicieron diferentes usuarios luego de una experiencia de compras en Amazon.

Se trata de un conjunto de datos de revisiones de productos de Amazon para la clasificación de texto multilingüe. Contiene reseñas en inglés, japonés, alemán, francés, chino y español, recopiladas entre el 1 de noviembre de 2015 y el 1 de noviembre de 2019. Cada registro en el conjunto de datos contiene el texto de la revisión, el título de la revisión, la calificación por estrellas, un ID de revisor anonimizado, un ID de producto anonimizado y la categoría de producto de grano grueso (por ejemplo, 'libros', 'electrodomésticos', etc.). El corpus está equilibrado en estrellas, de modo que cada calificación por estrellas constituye el 20% de las revisiones en cada idioma.

Para cada idioma, hay 200,000 reseñas en los conjuntos de entrenamiento, desarrollo y prueba, respectivamente. El número máximo de revisiones por revisor es 20 y el número máximo de revisiones por producto es 20. Todas las revisiones se truncaron después de 2,000 caracteres, y todas las revisiones tienen al menos 20 caracteres de longitud.

Es importante tener en cuenta que el idioma de una revisión no necesariamente coincide con el idioma de su mercado (por ejemplo, las revisiones de amazon.de están principalmente escritas en alemán, pero también podrían estar escritas en inglés, etc.). Se aplicó un algoritmo de detección de idioma para determinar el idioma del texto de la revisión y eliminamos las revisiones que no estaban escritas en el idioma esperado.

De ese dataset original, vamos a trabajar con una muestra del 10%, es decir, de 20.000 reseñas y solamente nos vamos a quedar con los siguientes campos:

- `review_id`: Un character que identifica la reseña
- `stars`: Este campo fue trabajado, eliminando las reseñas de puntaje 3 (neutrales) y considerando las reseñas de 1 y 2 estrellas como "negativas" y las de 4 y 5 como "positivas".
- `review_body`: El texto de la reseña
- `product_category`: Un character que representa la categoría del producto

Vamos, entonces, a entrenar un modelo de clasificación que es una variación de la regresión logística (y lineal) que se llama "regresión regularizada por LASSO". Si bien no vamos a entrar en detalles solo diremos que Lla precisión predictiva de un modelo de regresión puede ser incrementada a través del encogimiento de los valores de los coeficientes o, incluso, haciéndolos cero.

Haciendo esto, se introduce algún sesgo pero se reduce la variancia de los valores pre-dichos y, por lo tanto, se incrementa la precisión predictiva total. En muchos casos, cuando existen muchos predictores puede ser necesario identificar un subconjunto más pequeño de estos predictores que muestren los efectos más grandes.Es por ello que resulta útil imponer restricciones en el proceso de estimación. A esta operación se la define como “regularización”.

Existen varios métodos de regularización  de  modelos  lineales:  non negative  garrotte  (Breiman,1995);  ridge  regression  (Hoerl y Kennard, 1970). Este trabajo se centrará en el LASSO. Este método utiliza la norma l1 como medida de penalización para definir las restricciones al modelo lineal. LASSO busca minimizar la siguiente expresión:

$$\sum_{i=1}^{n} (Y_{i} - \beta_{0} - \sum_{j=1}^{p} \beta_{j})^2 + \lambda  \sum_{j=1}^{p} |\beta_{j}| = RSS + \lambda  \sum_{j=1}^{p} |\beta_{j}|$$

Se parte de la minimización de la RSS clásico de la regresión y se agrega una restricción: el segundo término $\lambda  \sum_{j=1}^{p} |\beta_{j}|$ se hace pequeño cuando los coeficientes son pequeños y, por lo tanto, tiene el efecto de reducir los coeficientes $\beta$ estimados. $\lambda$ constituye un parámetro de tunning y su función es controlar el impacto relativo de ambos términos. Cuando el coeficiente es igual a cero ($\lambda=0$), LASSO es equivalente a un modelo lineal estimado por MCO. Por el contrario, a medida que el parámetro se hace más grande ($\lambda$), el término de restricción lo hace en igual proporción y todos los coeficientes se reducen para po-der satisfacer dicha restricción. En el límite, cuando λ es lo suficientemente grande todos los coeficientes se hacen igual a cero y solamente queda como parámetro el intercepto ($\beta_{0}$), es decir, algo casi equivalente a predecir y  solamente con la media de la distribución.

En nuestro caso, se trata de una regresión logística, pero cambia solamente la función de pérdida.

Vamos a analizar dos casos: uno, utilizando como features o predictores las variables generadas por una vectorización tipo TF-IDF. En el segundo caso, lo haremos con el uso de features derivadas de embeddings pre-entrenados.


# TF con LASSO

# Preparación inicial
El código comienza importando varias bibliotecas de Python esenciales para el análisis de datos y machine learning, incluyendo pandas para manipulación de datos, numpy para operaciones numéricas, y scikit-learn para machine learning.

# Carga y limpieza de datos
- Se carga un conjunto de datos de reseñas de Amazon desde un archivo CSV
- Se elimina la columna 'product_category' por no ser necesaria
- La columna 'stars' (estrellas) se convierte a tipo categórico

# Preprocesamiento de texto
Se define una función `preprocess_text` que realiza varias transformaciones en el texto:
1. Convierte todo el texto a minúsculas
2. Elimina signos de puntuación
3. Reemplaza números con la palabra 'DIGITO'
4. Elimina caracteres no ASCII (acentos, caracteres especiales, etc.)

# Preparación para el modelo
- Se aplica el preprocesamiento a la columna 'review_body' que contiene el texto de las reseñas
- Se preparan las variables para el modelo:
  - `X`: contiene el texto preprocesado de las reseñas
  - `y`: se crea una variable binaria donde:
    - 1 representa reseñas negativas
    - 0 representa reseñas no negativas

Este código está preparando los datos para un modelo de clasificación binaria que determinará si una reseña es negativa o no basándose en el texto de la misma. Las bibliotecas importadas sugieren que se utilizará regresión logística con vectorización TF-IDF para el texto, y se evaluará el modelo usando validación cruzada y varias métricas de rendimiento.

In [ ]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_path = 'https://raw.githubusercontent.com/gefero/ecyt_lcd_intro_nlp/main/U4/data/amazon_reviews_train_sample.csv'

In [ ]:
# Cargamos los datos
reviews = pd.read_csv(data_path)
reviews = reviews.drop('product_category', axis=1)

# Convertimos 'stars' a variable categórica
reviews['stars'] = reviews['stars'].astype('category')

# Función para preprocesar el texto
def preprocess_text(text):
    # Convertir a minúsculas
    text = text.lower()

    # Reemplazar puntuación
    text = re.sub(r'[^\w\s]', ' ', text)

    # Reemplazar números por 'DIGITO'
    text = re.sub(r'\d+', 'DIGITO', text)

    # Reemplazar caracteres no ASCII
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

    return text

# Aplicamos el preprocesamiento
reviews['review_body'] = reviews['review_body'].apply(preprocess_text)

# Dividimos los datos en entrenamiento y prueba
X = reviews['review_body']
y = (reviews['stars'] == 'Negativa').astype(int)  # Convertimos a binario


# División de datos
El código comienza dividiendo los datos en conjuntos de entrenamiento y prueba usando `train_test_split`:
- 75% para entrenamiento (X_train, y_train)
- 25% para prueba (X_test, y_test)
- Se usa `stratify=y` para mantener la misma proporción de clases en ambos conjuntos
- `random_state=664` asegura reproducibilidad

# Configuración del modelo
Se crea un pipeline que combina dos elementos:
1. **Vectorización TF**:
   - Convierte el texto en vectores numéricos
   - Usa un patrón de tokenización específico para palabras completas
   - Calcula una TFM con ngramas de 1 y 2 palabras

2. **Regresión Logística**:
   - Usa regularización L1 (LASSO)
   - Utiliza el solver 'liblinear' apropiado para L1
   - Se configura con random_state para reproducibilidad

# Búsqueda del mejor hiperparámetro
Se realiza una búsqueda del mejor valor de regularización (C):
- Genera 30 valores de C en escala logarítmica entre 10^-10 y 10^1
- Para cada valor de C:
  - Configura el modelo con ese valor
  - Realiza validación cruzada con 5 particiones (KFold)
  - Calcula el ROC AUC score
  - Almacena la media y desviación estándar del rendimiento

# Resultados
Finalmente:
- Convierte los resultados en un DataFrame
- Identifica el mejor valor de C basado en el ROC AUC más alto
- Imprime el mejor valor de C y su correspondiente rendimiento con intervalo de confianza

Este código está realizando una optimización de hiperparámetros para encontrar el nivel óptimo de regularización que mejore la capacidad del modelo para clasificar reseñas negativas y positivas.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=664,
    stratify=y
)

# Creamos una grilla de valores para el parámetro C (inverso de penalty)
# Nota: en sklearn, C = 1/penalty, por lo que valores más pequeños = más regularización
C_values = np.logspace(-10, 1, 30)

# Creamos el pipeline con TF y LASSO
pipeline = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1,3),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

# Configuramos la validación cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=234)

# Almacenaremos los resultados aquí
cv_results = []

In [ ]:
%%time
# Para cada valor de C
for C in C_values:
    pipeline.set_params(classifier__C=C)

    # Calculamos el ROC AUC con validación cruzada
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )

    cv_results.append({
        'C': C,
        'mean_roc_auc': scores.mean(),
        'std_roc_auc': scores.std()
    })

# Convertimos resultados a DataFrame
cv_results_df = pd.DataFrame(cv_results)

# Encontramos el mejor C
best_idx = cv_results_df['mean_roc_auc'].idxmax()
best_C = cv_results_df.loc[best_idx, 'C']

print(f"Mejor valor de C: {best_C:.6f}")
print(f"Mejor ROC AUC (CV): {cv_results_df.loc[best_idx, 'mean_roc_auc']:.3f} ± {cv_results_df.loc[best_idx, 'std_roc_auc']:.3f}")

Mejor valor de C: 0.727895
Mejor ROC AUC (CV): 0.943 ± 0.005
CPU times: user 5min 44s, sys: 583 ms, total: 5min 45s
Wall time: 5min 54s


# Creación del modelo final
Se crea un nuevo pipeline con la misma estructura que el anterior, pero usando el mejor valor de C encontrado en la búsqueda de hiperparámetros. Incluye:
- Vectorizador TF con ngramas de 1 y 2 tokens
- Regresión Logística con regularización L1 y el valor óptimo de C

# Entrenamiento y predicciones
1. Se entrena el modelo final usando los datos de entrenamiento (`X_train`, `y_train`)
2. Se realizan dos tipos de predicciones sobre el conjunto de prueba:
   - `predict`: predicciones binarias (0 o 1)
   - `predict_proba`: probabilidades de predicción

# Evaluación del modelo
Se calculan múltiples métricas de rendimiento:
- **ROC AUC**: Mide la capacidad del modelo para distinguir entre clases
- **Accuracy**: Proporción de predicciones correctas
- **Precision**: De las predicciones positivas, cuántas son correctas
- **Recall**: De los casos realmente positivos, cuántos fueron identificados
- **F1**: Media armónica entre precision y recall

Los resultados se imprimen mostrando el valor de cada métrica, lo que permite evaluar el rendimiento del modelo desde diferentes perspectivas. Esta evaluación final nos da una idea clara de qué tan bien funciona nuestro modelo en datos no vistos durante el entrenamiento.

In [ ]:
# Ajustamos el modelo final con el mejor C
final_pipeline = Pipeline([
    ('vectorizer', CountVectorizer(
        ngram_range=(1,3),
        token_pattern=r'\b\w+\b'
    )),
    ('classifier', LogisticRegression(
        C=best_C,
        penalty='l1',
        solver='liblinear',
        random_state=234
    ))
])

# Entrenamos el modelo final
final_pipeline.fit(X_train, y_train)

# Hacemos predicciones en el conjunto de prueba
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Calculamos las métricas finales
results = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

# Mostramos los resultados finales
for metric, value in results.items():
    print(f"{metric}: {value:.3f}")

roc_auc: 0.940
accuracy: 0.879
precision: 0.881
recall: 0.876
f1: 0.879


# Word embeddings como features

## Importaciones y preparación inicial
El código comienza importando las bibliotecas necesarias, incluyendo:
- gensim para manejar word embeddings
- nltk para procesamiento de lenguaje natural
- las bibliotecas estándar para machine learning

## Preprocesamiento de datos
A diferencia del enfoque anterior, aquí el preprocesamiento es más simple:
- Solo reemplaza números con la palabra 'DIGITO'
- Mantiene el caso original y la puntuación
- Utiliza NLTK para tokenización de palabras

## Manejo de word embeddings
Se implementa un sistema para cargar y utilizar embeddings preentrenados:
1. Carga un modelo de word embeddings en español (sbwce)
2. Cada palabra se representa como un vector de 300 dimensiones
3. Para palabras fuera del vocabulario, se ignoran en vez de tratarlas especialmente

## Vectorización de reseñas
La función `get_mean_vector` convierte cada reseña en un vector mediante:
1. Tokenización del texto en palabras individuales
2. Búsqueda del vector correspondiente a cada palabra
3. Cálculo del promedio de todos los vectores de palabras
4. Si no hay palabras válidas, devuelve un vector de ceros

## Preparación final de datos
- Convierte todas las reseñas a vectores de 300 dimensiones
- Crea un DataFrame con las características vectoriales
- Mantiene el ID de la reseña para trazabilidad
- Prepara la variable objetivo (y) como binaria (Negativa = 1, otras = 0)

La diferencia principal con el enfoque anterior es que aquí se utilizan word embeddings preentrenados en lugar de TF-IDF, lo que puede capturar mejor las relaciones semánticas entre palabras y el contexto del lenguaje.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import re
from gensim.models import KeyedVectors
import nltk
from nltk.tokenize import word_tokenize
import warnings
warnings.filterwarnings('ignore')
nltk.download('punkt_tab')

# Descarga necesaria para tokenización
nltk.download('punkt')

# Cargamos los datos
reviews = pd.read_csv(data_path)
reviews = reviews.drop('product_category', axis=1)
reviews['stars'] = reviews['stars'].astype('category')

# Preprocesamiento más simple (solo reemplazamos dígitos)
def preprocess_text(text):
    # Reemplazar números por 'DIGITO'
    text = re.sub(r'\d+', 'DIGITO', text)
    return text

reviews['review_body'] = reviews['review_body'].apply(preprocess_text)

# Cargar el modelo de word embeddings
def load_embeddings(path):
    print("Cargando embeddings...")
    return KeyedVectors.load_word2vec_format(path, binary=True)

# Carga el modelo (asegúrate de tener el archivo correcto)
word_vectors = load_embeddings("./models/SBW-vectors-300-min5.bin")

# Función para obtener el vector promedio de una review
def get_mean_vector(text, word_vectors, vector_size=300):
    words = word_tokenize(text.lower())  # Tokenizamos y convertimos a minúsculas
    word_vectors_list = []

    for word in words:
        try:
            vector = word_vectors[word]
            word_vectors_list.append(vector)
        except KeyError:
            continue  # Ignoramos palabras que no están en el embedding

    if word_vectors_list:
        return np.mean(word_vectors_list, axis=0)
    else:
        return np.zeros(vector_size)  # Vector de ceros si no hay palabras válidas

# Convertir reviews a vectores
print("Vectorizando reviews...")
review_vectors = []
for text in reviews['review_body']:
    review_vectors.append(get_mean_vector(text, word_vectors))

# Convertir a DataFrame
X = pd.DataFrame(review_vectors, columns=[f'V{i+1}' for i in range(300)])
X['review_id'] = reviews['review_id']
y = (reviews['stars'] == 'Negativa').astype(int)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Cargando embeddings...
Vectorizando reviews...


In [ ]:
X.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V292,V293,V294,V295,V296,V297,V298,V299,V300,review_id
0,0.003872,-0.100006,0.101065,-0.025584,0.036041,0.017133,0.000402,-0.054167,0.226962,-0.032584,...,0.093506,-0.058827,-0.063638,-0.062807,-0.166243,-0.065480,-0.182469,0.001289,0.068484,es_0631804
1,0.002346,-0.072056,0.081040,-0.081154,0.005422,0.004181,-0.052396,-0.083455,0.261127,-0.045779,...,0.058216,-0.119157,0.040346,-0.057988,-0.144656,-0.032395,-0.151811,-0.025604,0.059512,es_0837984
2,-0.017616,-0.016638,0.085777,-0.102646,-0.045355,0.041728,-0.020699,-0.089352,0.222077,-0.058747,...,0.098065,-0.074158,-0.033875,-0.043240,-0.156351,-0.024875,-0.133513,-0.010456,0.088224,es_0683391
3,-0.047323,-0.065557,0.134710,-0.084838,-0.015908,0.037168,-0.020217,-0.084752,0.259468,0.017777,...,0.057635,-0.091812,0.045776,-0.080494,-0.136458,-0.079081,-0.127683,0.000326,0.083922,es_0712963
4,-0.026055,-0.105232,0.093829,-0.047964,0.024159,0.016902,0.054569,-0.058275,0.223325,-0.021571,...,0.077802,-0.094030,-0.027993,-0.063891,-0.148188,0.016916,-0.160154,0.011474,0.015918,es_0222030


# División y preparación de datos
El código comienza dividiendo los datos en conjuntos de entrenamiento y prueba:
- Mantiene un 75% para entrenamiento y 25% para prueba
- Preserva los IDs de las reseñas separadamente
- Elimina la columna de IDs de los datos de entrenamiento y prueba
- Mantiene la estratificación para asegurar distribución similar de clases

# Optimización de hiperparámetros
Se realiza una búsqueda exhaustiva del mejor valor de regularización:
1. Genera 30 valores de C en escala logarítmica
2. Para cada valor de C:
   - Crea un modelo de regresión logística con regularización L1
   - Realiza validación cruzada con 5 particiones
   - Calcula el ROC AUC para cada partición
   - Almacena la media y desviación estándar

# Entrenamiento del modelo final
Una vez encontrado el mejor valor de C:
1. Crea un nuevo modelo con el C óptimo
2. Entrena usando todos los datos de entrenamiento
3. Realiza predicciones en el conjunto de prueba, tanto binarias como probabilísticas

# Evaluación del rendimiento
Se calculan múltiples métricas de evaluación:
- ROC AUC: para la capacidad discriminativa general
- Accuracy: para la precisión global
- Precision: para la confiabilidad de las predicciones positivas
- Recall: para la capacidad de detectar casos positivos
- F1: para el balance entre precision y recall

La principal diferencia con el enfoque TF-IDF es que aquí trabajamos directamente con los vectores densos de los word embeddings, lo que puede capturar mejor las relaciones semánticas entre palabras, aunque el proceso de modelado y evaluación sigue siendo similar.

In [ ]:
# División en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=664,
    stratify=y
)

# Separar review_id
train_ids = X_train['review_id']
test_ids = X_test['review_id']
X_train_embed = X_train.drop('review_id', axis=1)
X_test_embed = X_test.drop('review_id', axis=1)

# Creamos una grilla de valores para C
C_values = np.logspace(-10, 1, 30)

# Configuramos la validación cruzada
cv = KFold(n_splits=5, shuffle=True, random_state=234)

# Almacenaremos los resultados aquí
cv_results = []

In [ ]:
%%time
# Para cada valor de C
print("Realizando validación cruzada...")
for C in C_values:
    model = LogisticRegression(
        C=C,
        penalty='l1',
        solver='liblinear',
        random_state=234
    )

    # Calculamos el ROC AUC con validación cruzada
    scores = cross_val_score(
        model,
        X_train_embed,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )

    cv_results.append({
        'C': C,
        'mean_roc_auc': scores.mean(),
        'std_roc_auc': scores.std()
    })

# Convertimos resultados a DataFrame
cv_results_df = pd.DataFrame(cv_results)

# Encontramos el mejor C
best_idx = cv_results_df['mean_roc_auc'].idxmax()
best_C = cv_results_df.loc[best_idx, 'C']

print(f"\nMejor valor de C: {best_C:.6f}")
print(f"Mejor ROC AUC (CV): {cv_results_df.loc[best_idx, 'mean_roc_auc']:.3f} ± {cv_results_df.loc[best_idx, 'std_roc_auc']:.3f}")

Realizando validación cruzada...

Mejor valor de C: 1.743329
Mejor ROC AUC (CV): 0.913 ± 0.004
CPU times: user 2min 25s, sys: 6.16 s, total: 2min 31s
Wall time: 2min 16s


In [ ]:
# Ajustamos el modelo final con el mejor C
final_model = LogisticRegression(
    C=best_C,
    penalty='l1',
    solver='liblinear',
    random_state=234
)

# Entrenamos el modelo final
final_model.fit(X_train_embed, y_train)

# Hacemos predicciones en el conjunto de prueba
y_pred = final_model.predict(X_test_embed)
y_pred_proba = final_model.predict_proba(X_test_embed)[:, 1]

# Calculamos las métricas finales
results_embed = {
    'roc_auc': roc_auc_score(y_test, y_pred_proba),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print("\nResultados finales:")
for metric, value in results_embed.items():
    print(f"{metric}: {value:.3f}")


Resultados finales:
roc_auc: 0.909
accuracy: 0.836
precision: 0.826
recall: 0.852
f1: 0.839


In [ ]:
results_df = pd.DataFrame([results], index=['TF-IDF'])
results_embed_df = pd.DataFrame([results_embed], index=['Word Embeddings'])

comparison_df = pd.concat([results_df, results_embed_df])

print("Comparación de resultados de los modelos:")
display(comparison_df)

Comparación de resultados de los modelos:


,roc_auc,accuracy,precision,recall,f1
TF-IDF,0.939521,0.8792,0.881335,0.8764,0.878861
Word Embeddings,0.909444,0.8362,0.825902,0.8520,0.838748


# Ejercicio
Entrenar y tunear un Random Forest con features construidas mediante TF-IDF y otro con embeddings.

¿Cuál resulta más eficiente? ¿Por qué?

In [ ]:
###